In [ ]:
# ssGSEA for KEGG MEDICUS & GO_BP


suppressPackageStartupMessages({
  pkgs <- c("GSVA", "msigdbr", "dplyr", "ggplot2", "forcats",
            "scales", "GSEABase", "purrr")
  for(p in pkgs){
    if(!requireNamespace(p, quietly = TRUE)) install.packages(p)
    library(p, character.only = TRUE)
  }
})


# STEP 1: Data Preparation 
cat("\n[1] Preparing Expression Matrix & Metadata...\n")
file_path <- "TCGA_COAD_FINAL_WITH_RISK_D.csv" 

data <- read.csv(file_path, header = TRUE, row.names = 1, check.names = FALSE)
names(data) <- trimws(names(data))
names(data) <- gsub("\\.", "_", names(data))

rs_col <- grep("^risk.?score$", names(data), ignore.case = TRUE, value = TRUE)[1]
rg_col <- grep("^risk.?group$", names(data), ignore.case = TRUE, value = TRUE)[1]


meta <- data[, c(rs_col, rg_col)]
names(meta) <- c("RiskScore", "Risk_group")

meta$Risk_group <- trimws(toupper(as.character(meta$Risk_group)))
meta$Risk_group <- gsub(" RISK", "", meta$Risk_group, ignore.case = TRUE)
meta$Risk_group <- ifelse(meta$Risk_group == "LOW", "Low",
                          ifelse(meta$Risk_group == "HIGH", "High", NA))
meta$Risk_group <- factor(meta$Risk_group, levels = c("Low","High"))

expr <- data[, !(colnames(data) %in% c(rs_col, rg_col))]
expr <- expr[, sapply(expr, is.numeric)]
expr_matrix <- as.matrix(expr)

for(i in seq_len(ncol(expr_matrix))) {
  if(anyNA(expr_matrix[,i])) {
    expr_matrix[is.na(expr_matrix[,i]), i] <- median(expr_matrix[,i], na.rm = TRUE)
  }
}
expr_t <- t(expr_matrix)
cat(" ✓ Matrix & metadata prepared perfectly.\n")
print(table(meta$Risk_group))


cat(" STARTING KEGG MEDICUS PATHWAY ANALYSIS\n")

m_df <- msigdbr(species = "Homo sapiens", collection = "C2", subcollection = "CP:KEGG_MEDICUS")

gsc_kegg <- m_df %>%
  dplyr::select(gs_name, gene_symbol) %>%
  dplyr::filter(!is.na(gene_symbol), gene_symbol != "") %>%
  dplyr::group_by(gs_name) %>%
  dplyr::summarise(genes = list(unique(gene_symbol)), .groups = "drop") %>%
  purrr::pmap(function(gs_name, genes) { GeneSet(genes, setName = gs_name) }) %>%
  GeneSetCollection()

cat(" ✓ KEGG pathways loaded:", length(gsc_kegg), "\n")
cat(" Running ssGSEA for KEGG...\n")
ssgsea_param_kegg <- ssgseaParam(exprData = expr_t, geneSets = gsc_kegg, minSize = 10, maxSize = 500)
ssgsea_scores_kegg <- gsva(ssgsea_param_kegg, verbose = FALSE)
write.csv(as.data.frame(ssgsea_scores_kegg), "ssGSEA_KEGG_scores_matrix.csv", row.names = TRUE)

# -- KEGG Statistics
ssgsea_df_kegg <- as.data.frame(t(ssgsea_scores_kegg))
ssgsea_df_kegg$Risk_group <- meta$Risk_group
ssgsea_df_kegg <- ssgsea_df_kegg[!is.na(ssgsea_df_kegg$Risk_group), ]

pathway_stats_kegg <- lapply(setdiff(names(ssgsea_df_kegg), "Risk_group"), function(pw) {
  high_scores <- ssgsea_df_kegg[ssgsea_df_kegg$Risk_group == "High", pw]
  low_scores  <- ssgsea_df_kegg[ssgsea_df_kegg$Risk_group == "Low", pw]
  t_result <- tryCatch(t.test(high_scores, low_scores), error = function(e) NULL)
  
  if (is.null(t_result)) return(data.frame(Pathway = pw, logFC = NA, Pvalue = NA))
  
  m_high <- mean(high_scores, na.rm = TRUE)
  m_low <- mean(low_scores, na.rm = TRUE)
  data.frame(Pathway = pw, logFC = log2((m_high + 1e-6)/(m_low + 1e-6)), Pvalue = t_result$p.value)
}) %>% bind_rows() %>% filter(!is.na(Pvalue)) %>% mutate(FDR = p.adjust(Pvalue, method = "BH"))

write.csv(pathway_stats_kegg, "KEGG_Pathway_Statistics_HighVsLow.csv", row.names = FALSE)

# -- KEGG Plotting Function (With P-values)
create_kegg_plot <- function(data, sig_col = "Pvalue", sig_cutoff = 0.05, max_pathways = 30) {
  sig_data <- data %>% filter(!!sym(sig_col) < sig_cutoff) %>% arrange(desc(abs(logFC)))
  if (nrow(sig_data) == 0) sig_data <- data %>% arrange(!!sym(sig_col)) %>% slice_head(n = 15)
  else if (nrow(sig_data) > max_pathways) sig_data <- sig_data %>% slice_head(n = max_pathways)
  
  sig_data$Pathway_clean <- tools::toTitleCase(tolower(gsub("_", " ", gsub("^KEGG_MEDICUS_", "", sig_data$Pathway))))
  sig_data$neg_log_sig <- -log10(sig_data[[sig_col]])
  sig_data$Effect_Size <- abs(sig_data$logFC)
  
  # P-value
  sig_data <- sig_data %>%
    mutate(pval_label = ifelse(Pvalue < 0.001, "p < 0.001", sprintf("p = %.3f", Pvalue)))

  n_pathways <- nrow(sig_data)
  final_height <- max(7, min((n_pathways * 0.38) + 2.5, 25))
  final_width <- ifelse(mean(nchar(sig_data$Pathway_clean)) > 50, 16, 14)
  
  sig_expr <- ifelse(sig_col == "FDR", expression(-log[10]*"(FDR)"), expression(-log[10]*"(P-value)"))
  
  # Dot Plot
  p_dot <- ggplot(sig_data, aes(x = neg_log_sig, y = fct_reorder(Pathway_clean, neg_log_sig))) +
    geom_point(aes(size = Effect_Size, color = logFC), alpha = 0.85) +
    geom_text(aes(label = pval_label), hjust = -0.3, size = 3.5, fontface = "italic", color = "black") + # اضافه شدن P-value
    scale_color_gradient2(low = "#2166AC", mid = "gray95", high = "#B2182B", midpoint = 0, name = "log₂FC") +
    scale_size_continuous(name = "|log₂FC|", range = c(4, 12)) +
    scale_x_continuous(expand = expansion(mult = c(0.05, 0.2))) + # فضای کافی برای نوشته
    labs(title = sprintf("KEGG Pathway Enrichment (%s < %.2f)", sig_col, sig_cutoff), x = sig_expr, y = NULL) +
    theme_bw(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 15), panel.grid.minor = element_blank(), panel.grid.major.y = element_line(linetype = "dashed"))
  
  # Lollipop Plot
  p_lolli <- ggplot(sig_data, aes(x = logFC, y = fct_reorder(Pathway_clean, logFC))) +
    geom_segment(aes(x = 0, xend = logFC, yend = Pathway_clean), color = "gray70", linewidth = 1) +
    geom_point(aes(size = neg_log_sig, color = logFC), alpha = 0.9) +
    geom_text(aes(label = pval_label, hjust = ifelse(logFC >= 0, -0.4, 1.4)), size = 3.5, fontface = "italic", color = "black") + # اضافه شدن P-value
    scale_color_gradient2(low = "#3288BD", mid = "white", high = "#D53E4F", midpoint = 0, name = "log₂FC") +
    scale_size_continuous(name = sig_expr, range = c(4, 12)) +
    scale_x_continuous(expand = expansion(mult = c(0.15, 0.15))) + # فضای کافی برای نوشته
    geom_vline(xintercept = 0, linetype = "dashed", color = "black", linewidth = 0.6) +
    labs(title = sprintf("KEGG Pathway Enrichment - Hallmark Style (%s)", sig_col), x = "log₂ Fold Change", y = NULL) +
    theme_bw(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 15), panel.grid.minor = element_blank())
  
  ggsave(sprintf("KEGG_%s_DotPlot.png", sig_col), p_dot, width = final_width, height = final_height, dpi = 320)
  ggsave(sprintf("KEGG_%s_Lollipop.png", sig_col), p_lolli, width = final_width, height = final_height, dpi = 320)
}

create_kegg_plot(pathway_stats_kegg, "Pvalue", 0.05)
create_kegg_plot(pathway_stats_kegg, "FDR", 0.05)
cat(" ✓ KEGG Plots generated successfully!\n")



cat(" STARTING GO BIOLOGICAL PROCESS ANALYSIS\n")

go_bp_df <- msigdbr(species = "Homo sapiens", category = "C5", subcategory = "BP")

gsc_go <- go_bp_df %>%
  dplyr::select(gs_name, gene_symbol) %>%
  dplyr::filter(!is.na(gene_symbol), gene_symbol != "") %>%
  dplyr::group_by(gs_name) %>%
  dplyr::summarise(genes = list(unique(gene_symbol)), .groups = "drop") %>%
  purrr::pmap(function(gs_name, genes) { GeneSet(genes, setName = gs_name) }) %>%
  GeneSetCollection()

cat(" ✓ GO BP pathways loaded:", length(gsc_go), "\n")
cat(" Running ssGSEA for GO BP...\n")
ssgsea_param_go <- ssgseaParam(exprData = expr_t, geneSets = gsc_go, minSize = 10, maxSize = 500)
ssgsea_scores_go <- gsva(ssgsea_param_go, verbose = FALSE)
write.csv(as.data.frame(ssgsea_scores_go), "ssGSEA_GO_BP_scores_matrix.csv", row.names = TRUE)

# -- GO Statistics
ssgsea_df_go <- as.data.frame(t(ssgsea_scores_go))
ssgsea_df_go$Risk_group <- meta$Risk_group
ssgsea_df_go <- ssgsea_df_go[!is.na(ssgsea_df_go$Risk_group), ]

pathway_stats_go <- lapply(setdiff(names(ssgsea_df_go), "Risk_group"), function(pw) {
  high_scores <- ssgsea_df_go[ssgsea_df_go$Risk_group == "High", pw]
  low_scores  <- ssgsea_df_go[ssgsea_df_go$Risk_group == "Low", pw]
  t_result <- tryCatch(t.test(high_scores, low_scores), error = function(e) NULL)
  
  if (is.null(t_result)) return(data.frame(Pathway = pw, logFC = NA, Pvalue = NA))
  
  m_high <- mean(high_scores, na.rm = TRUE)
  m_low <- mean(low_scores, na.rm = TRUE)
  data.frame(Pathway = pw, logFC = log2((m_high + 1e-6)/(m_low + 1e-6)), Pvalue = t_result$p.value)
}) %>% bind_rows() %>% filter(!is.na(Pvalue)) %>% mutate(FDR = p.adjust(Pvalue, method = "BH"))

write.csv(pathway_stats_go, "GO_BP_Pathway_Statistics.csv", row.names = FALSE)

# -- GO Plotting Function (With P-values)
create_GO_plot <- function(data, sig_col = "FDR", top_n = 25) {
  sig_data <- data %>% arrange(!!sym(sig_col)) %>% slice_head(n = top_n)
  sig_data$Clean <- gsub("_", " ", gsub("^GOBP_", "", sig_data$Pathway))
  sig_data$neglog <- -log10(sig_data[[sig_col]])
  
  sig_data <- sig_data %>%
    mutate(pval_label = ifelse(Pvalue < 0.001, "p < 0.001", sprintf("p = %.3f", Pvalue)))

  p <- ggplot(sig_data, aes(x = neglog, y = fct_reorder(Clean, neglog))) +
    geom_point(aes(size = abs(logFC), color = logFC), alpha = 0.85) +
    geom_text(aes(label = pval_label), hjust = -0.3, size = 3.5, fontface = "italic", color = "black") + # اضافه شدن P-value
    scale_color_gradient2(low = "#2166AC", mid = "white", high = "#B2182B", midpoint = 0, name = "log₂FC") +
    scale_size_continuous(range = c(3, 8), name = "|log₂FC|") +
    scale_x_continuous(expand = expansion(mult = c(0.05, 0.2))) + # فضای کافی برای نوشته
    labs(title = "Top GO Biological Process Pathways (ssGSEA)", x = expression(-log[10]*FDR), y = NULL) +
    theme_bw(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 15), panel.grid.minor = element_blank(), panel.grid.major.y = element_line(linetype = "dashed"))
    
  ggsave("GO_BP_Top25_DotPlot_with_Pval.png", p, width = 14, height = 9, dpi = 320)
  ggsave("GO_BP_Top25_DotPlot_with_Pval.pdf", p, width = 14, height = 9)
}

create_GO_plot(pathway_stats_go, "FDR", 25)
cat(" ✓ GO BP Plots generated successfully!\n")

